In [ ]:
%py
# PySpark script to test inserting a new record into the config_master table

from pyspark.sql.functions import col, lit
from pyspark.sql.utils import AnalysisException

# Load the config_master table for testing
config_master_df = spark.table("purgo_playground.config_master")

# Define a function to test transforming and appending a new record to config_master
def test_insert_config_master():
    try:
        # Step 1: Retrieve an existing record by src_objt_name
        existing_record_df = config_master_df.where(config_master_df.src_objt_name == "US_Sales").limit(1)
        
        # Check if any record is fetched
        assert existing_record_df.count() == 1, "No record found with src_objt_name 'US_Sales'"
        
        # Step 2: Transform the existing record with the specified new values
        new_record_df = existing_record_df.withColumn("src_objt_name", lit("ID_Sales")) \
                                          .withColumn("src_sys", lit("ID_Sales")) \
                                          .withColumn("f_format", lit("ID_MON_Sales_")) \
                                          .withColumn("s3_landing_path", lit("s3a://{s3_bucket}/landing/ID/ID_Sales/")) \
                                          .withColumn("s3_archive_path", lit("s3a://{s3_bucket}/archive/ID/ID_Sales/")) \
                                          .withColumn("country", lit("ID")) \
                                          .withColumn("region", lit("ID")) \
                                          .withColumn("affiliate_group", lit("ID")) \
                                          .withColumn("affiliate", lit("ID")) \
                                          .withColumn("src_layer", lit("ID_Sales")) \
                                          .withColumn("target_src_sys", lit("ID_Sales")) \
                                          .withColumn("delta_stg_tables", lit("stg_ID_sales")) \
                                          .withColumn("source_path", lit("/SecureFtp/-InternalX/ID/IN/DATA/Sales/")) \
                                          .withColumn("actual_file_name", lit('{"ID_MON_Sales_*": "stg_ID_wholesaler"}')) \
                                          .withColumn("dag_id", lit("LOAD_SALES_ID"))
        
        # Step 3: Ensure the config_id is unique by incrementing max ID
        max_config_id = config_master_df.agg({"config_id": "max"}).collect()[0][0]
        new_record_df = new_record_df.withColumn("config_id", lit(max_config_id + 1))
        
        # Step 4: Append the new record to the config_master table
        new_record_df.write.format("delta").mode("append").saveAsTable("purgo_playground.config_master")
        
        # Validate if the new record is inserted correctly
        inserted_record_count = spark.table("purgo_playground.config_master").where(col("src_objt_name")=="ID_Sales").count()
        assert inserted_record_count == 1, f"Expected 1 record to be inserted, found {inserted_record_count}"
        
        print("Test passed: New record inserted successfully in config_master.")
    
    except AnalysisException as ae:
        print(f"AnalysisException: {ae}")
    except Exception as e:
        print(f"Test failed with exception: {e}")

# Execute the test function
test_insert_config_master()

